In [1]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import mlflow

from src import feature_engineering as fe
from src import interpretation as interpret

/Users/hector.vargas/repos/ml_hands_on_project/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
X_train = pd.read_csv("../data/processed/raw_features/X_train.csv")
X_test = pd.read_csv("../data/processed/raw_features/X_test.csv")

RANDOM_STATE = 42
N_SPLITS = 5
EXPERIMENT_NAME = "customer-churn-optuna"

from pathlib import Path

def _find_project_root():
    """Find the project root by looking for pyproject.toml."""
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("Could not find project root (pyproject.toml)")

ROOT_DIR = str(_find_project_root())
DB_PATH = os.path.join(ROOT_DIR, "mlflow.db")
ARTIFACTS_DIR = os.path.join(ROOT_DIR, "mlartifacts")

# Set the tracking URI immediately to lock it to SQLite
mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")

In [3]:
selected_features = [
    # Binary
    "is_silver",
    "is_germany",
    "is_spain",
    "Num_Of_Products_1",
    "Num_Of_Products_2",
    "Num_Of_Products_3",
    "Num_Of_Products_4",

    # Continuous
    "Age_x_IsActive",
    "Balance_per_Product",
    "CreditScore_per_Age",
    "Inactive_x_Balance",
    "CreditScore_x_Age",
    "Products_per_Tenure",
]
# Schema Baseline Columns Definitions
nomod_columns = []
dummyfy_columns = ['Gender']
norm_std_columns = ['Point Earned', 'Satisfaction Score', 'EstimatedSalary']

# Initialize the Feature Engineer class with the desired subset strings
feature_engineer_object = fe.DynamicFeatureEngineer(selected_features=selected_features)
binary_features = feature_engineer_object._get_all_binary_features()
continuous_features = feature_engineer_object._get_all_continuous_features()

current_layout = {
    "passthrough": nomod_columns + binary_features,
    "standard_scale": norm_std_columns + continuous_features,
    "one_hot_encode": dummyfy_columns
}

EXPERIMENT_REGISTRY = {
    "experiment_1": current_layout
}

In [6]:
# Active the experiment scope
mlflow.set_experiment(EXPERIMENT_NAME)
experiment_id = "1"  # Your experiment ID

# 2. Programmatically query your tracking history to isolate the parent run
runs_df = mlflow.search_runs(experiment_ids=[experiment_id])

# Filter out the children; find the most recent successful parent run
parent_runs = runs_df[runs_df["tags.mlflow.parentRunId"].isna() & (runs_df["status"] == "FINISHED")]
best_parent_run = parent_runs.sort_values(by="metrics.test_pr_auc", ascending=False).iloc[0]

best_run_id = best_parent_run["run_id"]
best_pr_auc = best_parent_run["metrics.test_pr_auc"]

print(f"Fetching Champion Model from Parent Run ID: {best_run_id}")
print(f"Validated Validation PR-AUC: {best_pr_auc:.4f}")

# 3. Load the model artifact directly back into your environment
model_uri = f"runs:/{best_run_id}/best_model"
loaded_pipeline = mlflow.sklearn.load_model(model_uri)

print("\nPipeline successfully loaded! Type:", type(loaded_pipeline))

Fetching Champion Model from Parent Run ID: 5f2fd1dc86194a0f89a9555b199fa9cf
Validated Validation PR-AUC: 0.7333



Pipeline successfully loaded! Type: <class 'sklearn.pipeline.Pipeline'>


In [7]:
# Create a dummy DataFrame representing new raw production client data
new_customers = pd.DataFrame([{
    "Card Type": "SILVER",
    "Geography": "Germany",
    "Gender": "Female",
    "Balance": 52000.0,
    "Point Earned": 450,
    "CreditScore": 680,
    "Age": 38,
    "Tenure": 4,
    "Satisfaction Score": 4,
    "EstimatedSalary": 85000.0,
    "NumOfProducts": 2,
    "HasCrCard": 1,
    "IsActiveMember": 1
}])

# Generate classifications (0 = Retained, 1 = Churn)
predictions = loaded_pipeline.predict(new_customers)

# Generate raw prediction probability score arrays
probabilities = loaded_pipeline.predict_proba(new_customers)[:, 1]

print(f"Prediction Target Assignment: {predictions[0]}")
print(f"Calculated Churn Probability: {probabilities[0]:.2%}")

Prediction Target Assignment: 0
Calculated Churn Probability: 11.03%


In [ ]:
def extract_pipeline_components(pipeline):
    """
    Splits the fitted sklearn pipeline into its major stages.
    """
    fe_step = pipeline.named_steps["feature_engineering"]
    preprocessor = pipeline.named_steps["preprocessing"]
    model = pipeline.named_steps["model"]

    return fe_step, preprocessor, model

def print_model_details(best_run, pipeline):
    """
    Prints MLflow metadata and model hyperparameters.
    """

    print("\n========================")
    print("BEST RUN")
    print("========================")

    print(f"Run ID: {best_run.run_id}")

    if "metrics.average_precision" in best_run:
        print(
            f"Average Precision: "
            f"{best_run['metrics.average_precision']:.5f}"
        )

    model = pipeline.named_steps["model"]

    print("\n========================")
    print("MODEL PARAMETERS")
    print("========================")

    for k, v in model.get_params().items():
        print(f"{k}: {v}")

In [ ]:
def transform_raw_data(
    pipeline,
    X_raw: pd.DataFrame
):
    """
    Applies feature engineering + preprocessing.
    Returns transformed dataframe with labels.
    """

    fe_step = pipeline.named_steps["feature_engineering"]
    preprocessor = pipeline.named_steps["preprocessing"]

    X_engineered = fe_step.transform(X_raw)

    X_transformed = preprocessor.transform(X_engineered)

    feature_names = (
        preprocessor.get_feature_names_out()
    )

    X_transformed_df = pd.DataFrame(
        X_transformed,
        columns=feature_names,
        index=X_raw.index,
    )

    return X_transformed_df

In [ ]:
def predict_customer_churn(
    pipeline,
    X_raw: pd.DataFrame,
    threshold: float = 0.5,
):
    """
    Predict churn probability from raw customer data.
    """

    probabilities = pipeline.predict_proba(X_raw)[:, 1]

    predictions = (
        probabilities >= threshold
    ).astype(int)

    return pd.DataFrame(
        {
            "churn_probability": probabilities,
            "prediction": predictions,
        },
        index=X_raw.index,
    )

In [ ]:
import mlflow

experiment = mlflow.get_experiment_by_name(
    "customer-churn-optuna"
)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.average_precision DESC"]
)

runs.head()

In [ ]:
best_per_model = (
    runs
    .sort_values("metrics.average_precision", ascending=False)
    .groupby("params.model")
    .head(1)
    .loc[:, [
        "params.model",
        "metrics.average_precision",
        "run_id"
    ]]
)

print(best_per_model)

In [ ]:

preds = predict_customer_churn(
    pipeline,
    X_test.head(10)
)

print(preds)

In [ ]:
import shap


def generate_shap_explanations(
    pipeline,
    X_raw: pd.DataFrame,
):
    """
    Generates SHAP values from raw data.
    """

    model = pipeline.named_steps["model"]

    X_transformed_df = transform_raw_data(
        pipeline,
        X_raw,
    )

    explainer = shap.TreeExplainer(model)

    shap_values = explainer(
        X_transformed_df
    )

    return (
        explainer,
        shap_values,
        X_transformed_df,
    )

In [ ]:
def plot_global_shap(
    shap_values,
    X_transformed_df,
):
    """
    Global feature importance.
    """

    shap.plots.beeswarm(
        shap_values,
        max_display=20,
    )

In [ ]:
def plot_shap_importance(
    shap_values,
):
    shap.plots.bar(
        shap_values,
        max_display=20,
    )

In [ ]:
def explain_customer(
    shap_values,
    row_idx=0,
):
    shap.plots.waterfall(
        shap_values[row_idx],
        max_display=15,
    )

In [ ]:
pipeline, best_run = (
    interpret.load_best_run_pipeline(
        experiment_name=EXPERIMENT_NAME,
        db_path=DB_PATH,
        order_by_metric="metrics.average_precision DESC",
    )
)

print_model_details(
    best_run,
    pipeline,
)

predictions = predict_customer_churn(
    pipeline,
    X_test.head(10),
)

print(predictions)

(
    explainer,
    shap_values,
    X_transformed_df,
) = generate_shap_explanations(
    pipeline,
    X_test.sample(
        500,
        random_state=42,
    ),
)

plot_shap_importance(
    shap_values
)

plot_global_shap(
    shap_values,
    X_transformed_df,
)

explain_customer(
    shap_values,
    row_idx=0,
)